# Taxis NYC: preparación de datos, modelos y métricas

**AIMLF1 · Módulo 2 · Actividad en parejas** · Integrantes: _(completar)_

Este notebook produce **todo lo que consumen la API y la app**:

| Parte | Qué hace | Archivos que genera |
|---|---|---|
| Datos | Serie horaria de viajes por zona | — |
| 1. Pronóstico | XGBoost con 9 cuantiles + reglas simples | `api/modelo_pronostico.ubj`, `api/demanda.parquet`, `api/zonas.csv`, `api/meta.json` |
| 2. Segmentación | K-Means + PCA | `app/data/zonas_segmentadas.csv`, `grupos.csv`, `silueta.csv` |
| 3. Decisión | Cuantil óptimo según costos | (se valida aquí, se usa en la app) |
| 4. Ficha del modelo | Métricas, cobertura, error por tipo de zona | `app/data/metricas.json`, `metricas_diarias.csv` |

Ejecútenlo de arriba abajo (Runtime → Run all). Las cifras de la ficha del modelo salen de las celdas de la Parte 4.

In [ ]:
# Colab ya trae casi todo; esto asegura XGBoost >= 2.0 (necesario para varios cuantiles)
%pip install -q "xgboost>=2.0" pandas pyarrow scikit-learn matplotlib

In [ ]:
import json
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

SEMILLA = 42
BASE = Path(".")
API = BASE / "api"
DATOS_APP = BASE / "app" / "data"
for carpeta in (API, DATOS_APP):
    carpeta.mkdir(parents=True, exist_ok=True)

## 0. Reglas comunes
Se fijan en un solo lugar para que todo el notebook use los mismos valores.

In [ ]:
URL_VIAJES = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
URL_ZONAS = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

INICIO_MES = pd.Timestamp("2024-01-01")
INICIO_PRUEBA = pd.Timestamp("2024-01-25")      # entrenamiento: hasta el 24 inclusive
INICIO_FUTURO = pd.Timestamp("2024-02-01")      # "mañana": sin datos reales
FIN_FUTURO = pd.Timestamp("2024-02-01 23:00")
FERIADO = pd.Timestamp("2024-01-15").date()     # Martin Luther King Jr.

MIN_VIAJES_PRONOSTICO = 3000
MIN_VIAJES_SEGMENTACION = 500

QS = [round(0.1 * i, 1) for i in range(1, 10)]                 # 0.1 ... 0.9
NOMBRES_Q = [f"p{int(round(q * 100))}" for q in QS]            # p10 ... p90
I50 = NOMBRES_Q.index("p50")

# Un lag menor a 24 h sería fuga de datos: el pronóstico se hace el día anterior.
VARIABLES = ["lag_24", "lag_48", "lag_167", "lag_168", "lag_169",
             "media_dia_ant", "hora_dia", "dia_semana", "feriado", "zona"]

## 1. Datos
Descargamos el mes de enero de 2024 (≈ 50 MB) y nos quedamos solo con lo que se usa.

In [ ]:
archivo = BASE / "yellow_tripdata_2024-01.parquet"
if not archivo.exists():
    urllib.request.urlretrieve(URL_VIAJES, archivo)

columnas = ["tpep_pickup_datetime", "PULocationID", "trip_distance", "fare_amount", "tip_amount"]
viajes = pd.read_parquet(archivo, columns=columnas).rename(columns={"tpep_pickup_datetime": "inicio"})
print(f"Filas del archivo: {len(viajes):,}")

# Viajes válidos: hora de inicio dentro de enero de 2024
viajes = viajes[(viajes["inicio"] >= INICIO_MES) & (viajes["inicio"] < INICIO_FUTURO)].copy()
print(f"Viajes válidos de enero de 2024: {len(viajes):,}")

# keep_default_na=False: si no, "N/A" (zona 265) se lee como dato faltante
zonas = pd.read_csv(URL_ZONAS, keep_default_na=False).rename(
    columns={"LocationID": "zona_id", "Borough": "distrito", "Zone": "nombre"})
es_desconocida = zonas["distrito"].isin(["Unknown", "N/A"]) | zonas["nombre"].isin(["Unknown", "N/A", "Outside of NYC"])
VALIDAS = set(zonas.loc[~es_desconocida, "zona_id"])

viajes_por_zona = viajes["PULocationID"].value_counts()
ZONAS_PRON = sorted(int(z) for z in viajes_por_zona[viajes_por_zona >= MIN_VIAJES_PRONOSTICO].index if z in VALIDAS)
ZONAS_SEG = sorted(int(z) for z in viajes_por_zona[viajes_por_zona >= MIN_VIAJES_SEGMENTACION].index if z in VALIDAS)
print(f"Zonas válidas en el mapa: {len(VALIDAS)} | se pronostican: {len(ZONAS_PRON)} | se segmentan: {len(ZONAS_SEG)}")

In [ ]:
# Mirada rápida a la demanda total por hora: hay un ciclo diario y uno semanal
total_hora = viajes.groupby(viajes["inicio"].dt.floor("h")).size()
fig, ax = plt.subplots(figsize=(12, 3))
total_hora.plot(ax=ax, lw=0.8)
ax.set_title("Viajes por hora en toda la ciudad, enero de 2024")
ax.set_ylabel("Viajes")
ax.axvline(INICIO_PRUEBA, color="crimson", ls="--", label="inicio de la prueba (25 ene)")
ax.legend()
plt.show()

### De viajes a serie de tiempo
1. Redondear la hora de inicio a la hora.
2. Contar viajes por zona y hora.
3. **Rellenar con 0** las horas sin viajes (todas las combinaciones zona-hora).
4. Agregar el 1 de febrero vacío (lo que se va a pronosticar).
5. Crear las variables con `groupby("PULocationID")...shift(...)` para no mezclar zonas.

In [ ]:
viajes["hora"] = viajes["inicio"].dt.floor("h")
conteo = viajes.groupby(["PULocationID", "hora"]).size().rename("viajes")

horas = pd.date_range(INICIO_MES, FIN_FUTURO, freq="h")
indice = pd.MultiIndex.from_product([ZONAS_PRON, horas], names=["PULocationID", "hora"])
serie = conteo.reindex(indice, fill_value=0).reset_index()          # orden: zona, luego hora
serie.loc[serie["hora"] >= INICIO_FUTURO, "viajes"] = np.nan        # el 1 de feb no tiene datos reales

print("Horas con cero viajes que se rellenaron:", int((serie["viajes"] == 0).sum()))
serie.head()

In [ ]:
# Variables: solo información disponible el día anterior
grupo_zona = serie.groupby("PULocationID")["viajes"]
for lag in (24, 48, 167, 168, 169):
    serie[f"lag_{lag}"] = grupo_zona.shift(lag)

# Promedio de viajes por hora del día anterior (día calendario)
serie["dia"] = serie["hora"].dt.normalize()
diario = (serie[serie["hora"] < INICIO_FUTURO]
          .groupby(["PULocationID", "dia"])["viajes"].mean().rename("media_dia_ant").reset_index())
diario["dia"] = diario["dia"] + pd.Timedelta(days=1)
serie = serie.merge(diario, on=["PULocationID", "dia"], how="left")

serie["hora_dia"] = serie["hora"].dt.hour
serie["dia_semana"] = serie["hora"].dt.dayofweek
serie["feriado"] = (serie["hora"].dt.date == FERIADO).astype(int)
# zona como categoría nativa de XGBoost; las categorías se fijan con TODAS las zonas pronosticables
serie["zona"] = pd.Categorical(serie["PULocationID"], categories=ZONAS_PRON)

entreno = serie[serie["hora"] < INICIO_PRUEBA].dropna(subset=["lag_169"])      # la primera semana no tiene lags
prueba = serie[(serie["hora"] >= INICIO_PRUEBA) & (serie["hora"] < INICIO_FUTURO)].copy()
futuro = serie[serie["hora"] >= INICIO_FUTURO].copy()
print(f"entrenamiento: {len(entreno):,} filas | prueba: {len(prueba):,} | futuro: {len(futuro):,}")
print("Nulos en las variables de la prueba y el futuro:", int(prueba[VARIABLES].isna().sum().sum()), int(futuro[VARIABLES].isna().sum().sum()))

## Parte 1 · Pronóstico con incertidumbre
Un solo modelo XGBoost con `reg:quantileerror` entrega los 9 cuantiles (10 % … 90 %).
Se evalúa **por tiempo** (entrena hasta el 24, prueba del 25 al 31), nunca con `train_test_split`.

In [ ]:
modelo = xgb.XGBRegressor(
    objective="reg:quantileerror",
    quantile_alpha=np.array(QS),
    n_estimators=250,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    tree_method="hist",
    enable_categorical=True,
    random_state=SEMILLA,
)
modelo.fit(entreno[VARIABLES], entreno["viajes"])

def predecir(m, X):
    """Cuantiles sin negativos y ordenados por fila (evita cuantiles cruzados)."""
    return np.sort(np.maximum(m.predict(X), 0), axis=1)

pred = predecir(modelo, prueba[VARIABLES])
real = prueba["viajes"].to_numpy()
print("forma de las predicciones:", pred.shape)

### Comparación con reglas simples
- **Igual que ayer:** `lag_24`
- **Igual que hace una semana:** `lag_168`

In [ ]:
prueba["ayer"] = prueba["lag_24"]
prueba["semana"] = prueba["lag_168"]

mae = lambda estimado: float(np.abs(estimado - real).mean())
mae_modelo = mae(pred[:, I50])
mae_ayer = mae(prueba["ayer"].to_numpy())
mae_semana = mae(prueba["semana"].to_numpy())

tabla = pd.DataFrame({"MAE (viajes/hora)": {"Modelo (mediana p50)": mae_modelo,
                                            "Igual que ayer": mae_ayer,
                                            "Igual que hace una semana": mae_semana}})
mejora_semana = 1 - mae_modelo / mae_semana
print(tabla.round(2))
print(f"\nEl modelo cambia el error en {-mejora_semana:+.1%} respecto a 'igual que hace una semana'")

In [ ]:
# Cobertura del rango 10 %-90 % (debería rondar 80 %) y calibración de cada cuantil
dentro = (real >= pred[:, 0]) & (real <= pred[:, -1])
cobertura = float(dentro.mean())
calibracion = pd.DataFrame({"cuantil": QS, "proporción real ≤ pronóstico": [(real <= pred[:, i]).mean() for i in range(9)]})
print(f"Cobertura del rango p10-p90: {cobertura:.1%} (objetivo 80 %)")
print(calibracion.round(3).to_string(index=False))

In [ ]:
# Un ejemplo visual: Midtown Center (zona 161) el 31 de enero
zona_ej = 161 if 161 in ZONAS_PRON else ZONAS_PRON[0]
m = (prueba["PULocationID"] == zona_ej) & (prueba["hora"].dt.date == pd.Timestamp("2024-01-31").date())
h = prueba.loc[m, "hora_dia"].to_numpy()
fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(h, pred[m.to_numpy(), 0], pred[m.to_numpy(), -1], alpha=0.25, label="rango 10 %-90 %")
ax.plot(h, pred[m.to_numpy(), I50], label="pronóstico (mediana)")
ax.plot(h, prueba.loc[m, "semana"], ls=":", label="hace una semana")
ax.scatter(h, prueba.loc[m, "viajes"], color="crimson", s=18, label="real")
ax.set_title(f"{zonas.set_index('zona_id').loc[zona_ej, 'nombre']} · 31 de enero")
ax.set_xlabel("Hora del día"); ax.set_ylabel("Viajes"); ax.legend()
plt.show()

### Guardar para la API
La API **no recalcula lags**: recibe las filas del 25 de enero al 1 de febrero con las variables listas.
Guardamos el modelo con `save_model` (formato `.ubj`) y comprobamos que, al cargarlo como lo hará la API,
las predicciones son idénticas.

In [ ]:
modelo.save_model(API / "modelo_pronostico.ubj")

demanda = serie.loc[serie["hora"] >= INICIO_PRUEBA, ["PULocationID", "hora", "viajes"] + [v for v in VARIABLES if v != "zona"]].copy()
demanda.to_parquet(API / "demanda.parquet", index=False)

zonas_pron = zonas[zonas["zona_id"].isin(ZONAS_PRON)][["zona_id", "nombre", "distrito"]]
zonas_pron.to_csv(API / "zonas.csv", index=False)

(API / "meta.json").write_text(json.dumps({"variables": VARIABLES, "cuantiles": NOMBRES_Q,
                                           "fecha_futura": str(INICIO_FUTURO.date())}, indent=2))

# Prueba tal como la hace la API: cargar, convertir zona a category con las mismas categorías, predecir
m2 = xgb.XGBRegressor()
m2.load_model(API / "modelo_pronostico.ubj")
d = pd.read_parquet(API / "demanda.parquet")
d["zona"] = pd.Categorical(d["PULocationID"], categories=sorted(d["PULocationID"].unique()))
pred_api = predecir(m2, d.loc[d["hora"] < INICIO_FUTURO, VARIABLES])
assert np.allclose(pred_api, pred, atol=1e-3), "La API no reproduciría el pronóstico del notebook"
print("OK: el modelo cargado desde archivo reproduce el pronóstico")
for f in ("modelo_pronostico.ubj", "demanda.parquet"):
    print(f"{f}: {(API / f).stat().st_size / 1e6:.1f} MB")

## Parte 2 · Segmentación de zonas
Variables que describen **cómo se usa** cada zona (una fila por zona con ≥ 500 viajes):
viajes por día (escala log), distancia, tarifa y propina medianas, % fin de semana y % de viajes en tres franjas.
Distancia y tarifa se calculan descartando viajes con valor cero o negativo.

In [ ]:
v = viajes[viajes["PULocationID"].isin(ZONAS_SEG)].copy()
v["hora_dia"] = v["inicio"].dt.hour
v["finde"] = v["inicio"].dt.dayofweek >= 5
v["f_6_10"] = v["hora_dia"].between(6, 9)                         # 6:00 a 9:59
v["f_16_20"] = v["hora_dia"].between(16, 19)                      # 16:00 a 19:59
v["f_22_5"] = (v["hora_dia"] >= 22) | (v["hora_dia"] <= 5)        # 22:00 a 5:59

g = v.groupby("PULocationID")
validos = v[(v["trip_distance"] > 0) & (v["fare_amount"] > 0)].groupby("PULocationID")
dias_mes = 31
feat = pd.DataFrame({
    "viajes_dia": g.size() / dias_mes,
    "dist_mediana": validos["trip_distance"].median(),
    "tarifa_mediana": validos["fare_amount"].median(),
    "propina_mediana": validos["tip_amount"].median(),
    "pct_finde": g["finde"].mean(),
    "pct_6_10": g["f_6_10"].mean(),
    "pct_16_20": g["f_16_20"].mean(),
    "pct_22_5": g["f_22_5"].mean(),
}).dropna()
feat["log_viajes_dia"] = np.log10(feat["viajes_dia"])

VARS_SEG = ["log_viajes_dia", "dist_mediana", "tarifa_mediana", "propina_mediana",
            "pct_finde", "pct_6_10", "pct_16_20", "pct_22_5"]
X = StandardScaler().fit_transform(feat[VARS_SEG])
print(f"{len(feat)} zonas a segmentar")
feat.describe().round(2).T

### ¿Cuántos grupos? Coeficiente de silueta para k = 2 a 7

In [ ]:
siluetas = []
for k in range(2, 8):
    etiquetas = KMeans(n_clusters=k, n_init=20, random_state=SEMILLA).fit_predict(X)
    siluetas.append({"k": k, "silueta": silhouette_score(X, etiquetas)})
siluetas = pd.DataFrame(siluetas)
K = int(siluetas.loc[siluetas["silueta"].idxmax(), "k"])
print(siluetas.round(3).to_string(index=False))
print(f"\nMejor silueta: k = {K}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(siluetas["k"], siluetas["silueta"], color=["crimson" if k == K else "steelblue" for k in siluetas["k"]])
ax.set_xlabel("k (número de grupos)"); ax.set_ylabel("Silueta"); ax.set_title("Elección de k")
plt.show()

**Cuidado:** K-Means numera los grupos al azar. Por eso los nombres **no** se asignan por número:
se asignan según el perfil de cada grupo (el de mayor distancia mediana son los aeropuertos, etc.).

In [ ]:
km = KMeans(n_clusters=K, n_init=20, random_state=SEMILLA).fit(X)
feat["cluster"] = km.labels_
perfil = feat.groupby("cluster")[VARS_SEG].mean()

def nombrar(perfil):
    """Nombra cada grupo por lo que lo distingue, sin depender del número que dio K-Means."""
    libres = set(perfil.index)
    nombres = {}
    def tomar(nombre, columna):
        if libres:
            ganador = perfil.loc[list(libres), columna].idxmax()
            nombres[ganador] = nombre
            libres.remove(ganador)
    tomar("Aeropuertos y viajes largos", "dist_mediana")
    tomar("Vida nocturna", "pct_22_5")
    tomar("Manhattan central", "log_viajes_dia")
    for i, c in enumerate(sorted(libres)):
        nombres[c] = "Barrios periféricos" if len(libres) == 1 else f"Barrios periféricos {chr(65 + i)}"
    return nombres

NOMBRE_GRUPO = nombrar(perfil)
feat["grupo"] = feat["cluster"].map(NOMBRE_GRUPO)
resumen = feat.groupby("grupo").agg(zonas=("cluster", "size"), viajes_dia=("viajes_dia", "mean"),
                                    dist_mediana=("dist_mediana", "mean"), tarifa_mediana=("tarifa_mediana", "mean"),
                                    propina_mediana=("propina_mediana", "mean"), pct_noche=("pct_22_5", "mean"),
                                    pct_finde=("pct_finde", "mean")).round(2)
resumen

In [ ]:
# Revisen que los nombres tengan sentido con los perfiles (si no, ajusten la función nombrar)
for nombre, fila in resumen.iterrows():
    print(f"{nombre}: {int(fila.zonas)} zonas, {fila.viajes_dia:,.0f} viajes/día, {fila.dist_mediana:.1f} mi, "
          f"{fila.pct_noche:.0%} de viajes de 22 a 5 h")

### PCA a dos componentes (solo para dibujar)

In [ ]:
pca = PCA(n_components=2, random_state=SEMILLA)
componentes = pca.fit_transform(X)
feat["pc1"], feat["pc2"] = componentes[:, 0], componentes[:, 1]
var_pca = pca.explained_variance_ratio_
print(f"Variación conservada: PC1 {var_pca[0]:.1%} + PC2 {var_pca[1]:.1%} = {var_pca.sum():.1%}")

fig, ax = plt.subplots(figsize=(7, 5))
for nombre, sub in feat.groupby("grupo"):
    ax.scatter(sub["pc1"], sub["pc2"], s=20 + sub["viajes_dia"] / 20, alpha=0.7, label=nombre)
ax.set_xlabel("Componente 1"); ax.set_ylabel("Componente 2"); ax.legend(); ax.set_title("Zonas en dos ejes (PCA)")
plt.show()

In [ ]:
# Perfil horario de cada zona: proporción de sus viajes en cada hora (para el mini gráfico de la tabla)
perfil_hora = pd.crosstab(v["PULocationID"], v["hora_dia"], normalize="index").reindex(columns=range(24), fill_value=0)
perfil_hora.columns = [f"h{h:02d}" for h in perfil_hora.columns]

seg = (feat.reset_index().rename(columns={"PULocationID": "zona_id", "pct_22_5": "pct_noche"})
       .merge(zonas[["zona_id", "nombre", "distrito"]], on="zona_id")
       .merge(perfil_hora.reset_index().rename(columns={"PULocationID": "zona_id"}), on="zona_id"))
seg = seg.drop(columns=["cluster", "log_viajes_dia"])
seg.to_csv(DATOS_APP / "zonas_segmentadas.csv", index=False)

def describir(nombre, fila):
    base = {"Aeropuertos y viajes largos": "Viajes largos y caros (aeropuertos o zonas lejanas), con pocas zonas pero mucho valor por viaje.",
            "Vida nocturna": "Zonas donde buena parte de la demanda ocurre de noche y de madrugada.",
            "Manhattan central": "Zonas céntricas de alto volumen: muchos viajes cortos repartidos durante el día."}
    texto = base.get(nombre, "Zonas de menor volumen, con viajes algo más largos y demanda más dispersa.")
    return (f"{texto} {int(fila.zonas)} zonas, {fila.viajes_dia:,.0f} viajes/día en promedio, "
            f"distancia mediana {fila.dist_mediana:.1f} mi y {fila.pct_noche:.0%} de viajes de noche.")

grupos = resumen.reset_index()
grupos["descripcion"] = [describir(r.grupo, r) for r in grupos.itertuples()]
grupos = grupos.rename(columns={"zonas": "n_zonas"}).sort_values("viajes_dia", ascending=False)
grupos.to_csv(DATOS_APP / "grupos.csv", index=False)
siluetas.to_csv(DATOS_APP / "silueta.csv", index=False)
grupos[["grupo", "descripcion"]]

## Parte 3 · Decisión: cuántos taxis enviar
Problema del vendedor de periódicos: si un pasajero perdido cuesta `C_p` y un taxi ocioso `C_o`, lo óptimo es
cubrir el cuantil

$$q^* = \frac{C_p}{C_p + C_o}$$

Con 40 y 10 dólares, $q^* = 0.80$. Suponemos que un taxi atiende un viaje por hora.

In [ ]:
def cuantil_optimo(costo_perdido, costo_ocioso):
    return costo_perdido / (costo_perdido + costo_ocioso)

def taxis_a_enviar(pred, q):
    """Interpola entre los cuantiles entregados por el modelo (p10...p90) y redondea hacia arriba."""
    posicion = (np.clip(q, QS[0], QS[-1]) - QS[0]) / 0.1
    bajo = int(np.floor(posicion))
    alto = min(bajo + 1, len(QS) - 1)
    peso = posicion - bajo
    return np.ceil((1 - peso) * pred[:, bajo] + peso * pred[:, alto])

def costo_total(enviados, real, costo_ocioso, costo_perdido):
    return float((np.maximum(enviados - real, 0) * costo_ocioso + np.maximum(real - enviados, 0) * costo_perdido).sum())

C_O, C_P = 10, 40
q_opt = cuantil_optimo(C_P, C_O)
print(f"Cuantil óptimo con Cp={C_P} y Co={C_O}: {q_opt:.2f}")

In [ ]:
# Toda la semana de prueba, todas las zonas, tres reglas de envío
reglas = {
    f"Recomendación (p{q_opt * 100:.0f})": taxis_a_enviar(pred, q_opt),
    "Pronóstico central (p50)": np.round(pred[:, I50]),
    "Igual que hace una semana": prueba["semana"].to_numpy(),
}
costos = pd.Series({n: costo_total(e, real, C_O, C_P) for n, e in reglas.items()})
print((costos.round(0)).to_string())
print(f"\nLa recomendación ahorra {1 - costos.iloc[0] / costos.iloc[1]:.0%} frente a enviar el p50 y {1 - costos.iloc[0] / costos.iloc[2]:.0%} frente a 'hace una semana'")

# Un día en una zona (como la página de la app)
m = ((prueba["PULocationID"] == zona_ej) & (prueba["hora"].dt.date == pd.Timestamp("2024-01-31").date())).to_numpy()
ej = {n: costo_total(e[m], real[m], C_O, C_P) for n, e in reglas.items()}
print("\nZona", zona_ej, "· 31 de enero:", {n: round(c) for n, c in ej.items()})

In [ ]:
# ¿Qué pasa si cambian los costos? El cuantil óptimo mueve la recomendación
filas = []
for c_p in (10, 20, 40, 80):
    q = cuantil_optimo(c_p, C_O)
    filas.append({"Cp": c_p, "Co": C_O, "cuantil óptimo": round(q, 2),
                  "costo recomendación": round(costo_total(taxis_a_enviar(pred, q), real, C_O, c_p)),
                  "costo enviar p50": round(costo_total(np.round(pred[:, I50]), real, C_O, c_p))})
pd.DataFrame(filas)

## Parte 4 · Ficha del modelo
Todas las cifras de la ficha salen de estas celdas y se guardan en `app/data/metricas.json`.

In [ ]:
prueba["grupo"] = prueba["PULocationID"].map(feat["grupo"])
prueba["p10"], prueba["p50"], prueba["p90"] = pred[:, 0], pred[:, I50], pred[:, -1]
prueba["err_modelo"] = (prueba["p50"] - prueba["viajes"]).abs()
prueba["err_ayer"] = (prueba["ayer"] - prueba["viajes"]).abs()
prueba["err_semana"] = (prueba["semana"] - prueba["viajes"]).abs()
prueba["dentro"] = (prueba["viajes"] >= prueba["p10"]) & (prueba["viajes"] <= prueba["p90"])

por_grupo = (prueba.groupby("grupo").agg(mae=("err_modelo", "mean"), mae_semana=("err_semana", "mean"),
                                          cobertura=("dentro", "mean"), viajes_hora=("viajes", "mean"),
                                          n_zonas=("PULocationID", "nunique")).round(3).sort_values("mae", ascending=False))
por_zona = prueba.groupby("PULocationID")["dentro"].mean()
por_grupo

In [ ]:
# Error día a día de la semana de prueba (base de la página extra de monitoreo)
prueba["fecha"] = prueba["hora"].dt.date
diario = (prueba.groupby("fecha").agg(mae_modelo=("err_modelo", "mean"), mae_semana=("err_semana", "mean"),
                                       cobertura=("dentro", "mean"), viajes_real=("viajes", "sum")).reset_index())
diario["fecha"] = diario["fecha"].astype(str)
diario.to_csv(DATOS_APP / "metricas_diarias.csv", index=False)
UMBRAL_ERROR = round(1.25 * mae_modelo, 1)     # alerta si un día supera 25 % el error promedio
print(f"Umbral de alerta sugerido: {UMBRAL_ERROR} viajes/hora")
diario.round(2)

In [ ]:
# ¿Quién queda fuera? Zonas válidas sin pronóstico (menos de 3,000 viajes en el mes)
fuera = zonas[zonas["zona_id"].isin(VALIDAS - set(ZONAS_PRON))]
viajes_fuera = viajes[viajes["PULocationID"].isin(fuera["zona_id"])].shape[0]
pct_fuera_manhattan = float((fuera["distrito"] == "Manhattan").mean())
print(f"{len(fuera)} zonas sin pronóstico ({viajes_fuera / len(viajes):.1%} de los viajes); "
      f"{1 - pct_fuera_manhattan:.0%} de ellas están fuera de Manhattan")
print(fuera["distrito"].value_counts().to_string())

In [ ]:
metricas = {
    "entrenamiento_hasta": "2024-01-24", "prueba": "2024-01-25 al 2024-01-31", "horizonte_horas": 24,
    "n_zonas_pronostico": len(ZONAS_PRON), "n_zonas_segmentadas": len(feat), "n_grupos": K,
    "n_horas_zona_prueba": int(len(prueba)), "n_viajes_enero": int(len(viajes)),
    "mae_modelo": mae_modelo, "mae_ayer": mae_ayer, "mae_semana": mae_semana,
    "mejora_vs_semana_pct": float(mejora_semana * 100),
    "cobertura_80": cobertura, "cobertura_min_zona": float(por_zona.min()), "cobertura_max_zona": float(por_zona.max()),
    "umbral_error": UMBRAL_ERROR,
    "n_zonas_sin_pronostico": int(len(fuera)), "pct_viajes_sin_pronostico": viajes_fuera / len(viajes),
    "pct_sin_pronostico_fuera_manhattan": 1 - pct_fuera_manhattan,
    "error_por_grupo": por_grupo.reset_index().to_dict("records"),
    "costos_semana_prueba": {"cp": C_P, "co": C_O, **{k: float(c) for k, c in costos.items()}},
    "segmentacion": {"k": K, "silueta": float(siluetas.loc[siluetas["k"] == K, "silueta"].iloc[0]),
                     "pca_pc1": float(var_pca[0]), "pca_pc2": float(var_pca[1])},
}
(DATOS_APP / "metricas.json").write_text(json.dumps(metricas, indent=2, ensure_ascii=False, default=float))

peor = por_grupo["mae"].idxmax()
print("RESUMEN PARA LA FICHA")
print(f"- Error medio del modelo: {mae_modelo:.1f} viajes/hora (ayer: {mae_ayer:.1f}, hace una semana: {mae_semana:.1f}).")
print(f"- Frente a 'hace una semana' el error cambia {-mejora_semana:+.1%}: "
      f"{'gana' if mae_modelo < mae_semana else 'no gana'}; el valor está en el rango, que la regla simple no da.")
print(f"- Cobertura del rango 10-90 %: {cobertura:.0%} (por zona de {por_zona.min():.0%} a {por_zona.max():.0%}).")
print(f"- Donde más se equivoca: '{peor}' ({por_grupo.loc[peor, 'mae']:.1f} viajes/hora).")

### Limitaciones y usos no recomendados (para la ficha)
- **Un solo mes de invierno:** no conoce el verano, vacaciones ni eventos.
- **Solo usa historia de la demanda y calendario:** no sabe de clima, conciertos ni cierres de calles.
- **Zonas sin pronóstico:** las zonas con menos de 3,000 viajes al mes (ver celda anterior) no tienen modelo.
- **No usar** para decidir qué conductores trabajan ni evaluarlos.
- **No usar para recortar servicio en barrios periféricos:** poca demanda de *taxi amarillo* no es poca necesidad de transporte
  (allí dominan los taxis verdes y las apps, que no están en estos datos).
- **No usar a más de un día:** está entrenado para el día siguiente.

In [ ]:
print("Archivos generados:")
for carpeta in (API, DATOS_APP):
    for f in sorted(carpeta.iterdir()):
        print(f"  {f.as_posix():45s} {f.stat().st_size / 1e6:6.2f} MB")

## Descargar los archivos desde Colab
Colab borra los archivos al cerrar la sesión. Esta celda los comprime y los descarga para subirlos al repositorio
(`api/` y `app/data/`).

In [ ]:
import shutil
shutil.make_archive("archivos_generados", "zip", root_dir=".", base_dir="api")
shutil.make_archive("archivos_app_data", "zip", root_dir="app", base_dir="data")
try:
    from google.colab import files
    files.download("archivos_generados.zip")
    files.download("archivos_app_data.zip")
except ImportError:
    print("Fuera de Colab: los archivos ya están en api/ y app/data/")